# fKF from the joint posterior, and the fKF that assumes the true noise

The fixed family of Section 3 of the new `main.tex`, the last of the four:

| family | $\tilde{\boldsymbol{\Sigma}}_t$ | draft | where |
|---|---|---|---|
| KF | full | eqs. (31)-(32) | `vkf-kf.ipynb` |
| vKF | diagonal | eqs. (33)-(34) | `vkf-kf.ipynb` |
| sKF | $\tilde v_t\boldsymbol{I}$ | eqs. (35)-(36) | notebook 07, `vkf-kf.ipynb` |
| fKF | $v\boldsymbol{I}$, fixed | eq. (37) | **here** |

Equation numbers are those of `main.tex` at overleaf commit `03e62fa`, with the label beside them
where it helps, since the numbers move when the draft changes.

Two questions.

1. **At rest.** The fKF from the joint posterior, eq. (37) (`fKF.mean`), on the setup of notebook 03,
   under both `b_eta` conventions. Notebook 04 has the same filter in the old marginal form, eq. (76)
   (`exact.fKF`) of `main_minorization.tex`, and only at $b_\eta = \sqrt{v_\eta/2}$.
2. **Across a change.** Section 4 of the draft says the filter that assumes the true noise
   ($\beta^* = 0.2$) recovers from a change 1.2 to 2.2 times more slowly than the Laplacian filters.
   Leszek's own runs (`sims/sep18/ggbench.py`, overleaf commit `03e62fa`) put the 2.2 on the **fKF**:
   2.14 against the exact Laplacian fKF, 2.22 against the minorized one, with white input. Notebook 09
   tested only the sKF, in our setup, and found the opposite (0.79). This notebook runs the fKF on
   notebook 09's sign-flip test.

Three fKFs, differing only in the correction $\chi_t$: the **minorized** Laplacian, eq. (45), the
**exact** Laplacian (joint), eq. (43), and the **matched** one, eq. (30) by quadrature, which assumes
the generalized Gaussian at $\beta^*$ and its true scale.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Notebook 03's, as in `vkf-kf.ipynb`: $M = 128$ room impulse response of section 7.1, AR($-0.9$)
input of unit variance, generalized Gaussian noise at $\beta^* = 0.2$ and SNR 5 dB, $N = 96000$,
$R = 20$, target misalignment $-20$ dB. Same seeds, so the same signals.

In [ ]:
# Copied from vkf-kf.ipynb, commit 2724d95 (notebook 03's scenario, AR coefficient as a knob).
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run, as in notebook 03
R = 20                      # independent realisations for the checked runs
R_SEARCH = 3                # realisations while scanning the grid
N_SEARCH = 24000            # scan length
WARMUP = 500                # AR samples discarded so the input starts stationary
AR_A = -0.9                 # AR(1) coefficient of the input
SNR_DB = 5.0                # as in Fig. 3 of the paper
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
VAR_THETA_0 = 2.0           # initial prior variance on each weight (the sKF control only)
TARGET_DB = -20.0           # target misalignment
FS = 8000
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]

ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM,
                  reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)


def correlation(ar_a, m=M):
    lags = np.abs(np.subtract.outer(np.arange(m), np.arange(m)))
    return ar_a**lags if ar_a != 0 else np.eye(m)


def b_eta_of(var_eta, convention):
    """The Laplacian scale the filter is told, under either convention (issue #9).

    "variance": b = sqrt(v_eta/2), matching the noise variance. What every notebook up to 07 used.
    "mad":      b = E|eta_t|, matching the mean absolute deviation. The maximum-likelihood and
                KL-closest Laplacian fit to the actual noise, and what the new draft uses."""
    if convention == "variance":
        return np.sqrt(var_eta/2)
    scale = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    return scale*np.exp(gammaln(2/BETA) - gammaln(1/BETA))


def scenario(ar_a, convention="variance"):
    """Signal power, noise variance and Laplacian scale at this input correlation."""
    P_signal = float(ho @ correlation(ar_a) @ ho)
    var_eta = P_signal/10**(SNR_DB/10)
    return P_signal, var_eta, b_eta_of(var_eta, convention)


def generate_signals(ar_a, seed, n=N):
    """generate_signals of notebook 03, with the AR coefficient as a knob (0 = white).

    The noise depends on the SNR alone, not on which b_eta the filter is told."""
    _, var_eta, _ = scenario(ar_a)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(N + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    scale = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    d = np.convolve(ho, x)[:N] + gennorm.rvs(BETA, scale=scale, size=N, random_state=rng)
    return x[:n], d[:n]
# === end of the copied block ===


_, VAR_ETA, B_VAR = scenario(AR_A, "variance")
_, _, B_MAD = scenario(AR_A, "mad")
print(f"AR a = {AR_A}, SNR {SNR_DB:.0f} dB, beta* = {BETA}")
print(f"   var_eta                  = {VAR_ETA:.5f}")
print(f"   b_eta = sqrt(var_eta/2)  = {B_VAR:.5f}   (old convention)")
print(f"   b_eta = E|eta|           = {B_MAD:.5f}   (new draft, issue #9)")

## 3. The three corrections

The fKF, eq. (37), is $\boldsymbol{w}_t = \boldsymbol{w}_{t-1} + \boldsymbol{x}_t\,\chi_t(e_t)/\|\boldsymbol{x}_t\|^2$ with
$\sigma_t^2 = v\|\boldsymbol{x}_t\|^2$, so the three filters differ only in $\chi_t$:

| filter | $\chi_t(e_t)$ | draft | source |
|---|---|---|---|
| minorized | $\tau_t e_t/(\tau_t + \lvert e_t\rvert)$, $\tau_t = \sigma_t^2/b_\eta$ | eq. (45) (`chi.minorized`), from the minorization (44); Section 3.5, Table 2 | written here |
| exact (joint) | $\tau_t\Lambda_t$ | eq. (43) (`Lap.chi`), $\Lambda_t$ from (41), $u_t$ and $k_t$ from (39) | `chi_laplacian`, notebook 07 |
| matched | mean of the scalar posterior, by quadrature | eq. (30) (`chi.def`), the mean of the scalar posterior (21), by the quadrature of Section 3.4 ("Heavier-tailed noise") | `chi_quadrature`, notebook 08 |

With the exact correction, the draft writes the fKF out right after (43):
$\boldsymbol{w}_t = \boldsymbol{w}_{t-1} + (v/b_\eta)\,\Lambda_t\,\boldsymbol{x}_t$. The generalized Gaussian
density of the matched filter has no numbered equation in the draft.

In [ ]:
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized,
#         notebooks/07_skf_conjunto.ipynb, commit 9406077 (via vkf-kf.ipynb)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


def chi_minorized(e, sigma, b_eta):
    """Eq. (45) of the draft (chi.minorized): the minorized correction tau e/(tau + |e|), and the
    ratio chi/e that replaces chi' in the variance update (Table 2). The fKF uses the first only."""
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta
    ratio = tau/(tau + np.abs(e))                      # chi_min / e, in (0, 1]
    return ratio*e, ratio


print("closed-form corrections ready")

In [ ]:
# === IGNACIO: gg_scale and the quadrature - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized,
#         notebooks/08_cuadratura.ipynb, commit 0ed92a4
def gg_scale(beta, var_eta):
    """alpha(beta): the generalized Gaussian of shape beta closest in KL to the simulated noise."""
    alpha_true = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))                 # alpha_*
    mean_abs_pow = alpha_true**beta*np.exp(gammaln((beta + 1)/BETA) - gammaln(1/BETA))    # E|eta|^beta
    return (beta*mean_abs_pow)**(1/beta)                                                   # alpha(beta)


L_WINDOW = 10               # the prior N(s; 0, sigma^2) is below e^{-50} beyond L_WINDOW sigma


def log_likelihood(u, density, params):
    """log p_eta(u) up to a constant: the noise density the filter assumes, in eqs. (21) and (24)."""
    if density == "gg":                                # generalized Gaussian, shape beta, scale alpha
        return -(np.abs(u)/params["alpha"])**params["beta"]
    # Student-t, nu degrees of freedom and scale c: -(nu + 1)/2 log(1 + (u/c)^2/nu)
    return -0.5*(params["nu"] + 1)*np.log1p((u/params["scale"])**2/params["nu"])


def piece_in_s(e, s_a, s_b, density, params):
    """Gauss-Legendre nodes on [s_a, s_b], directly in s. Returns the nodes and the log of
    (weight x length x likelihood p_eta(e_t - s)), the prior left out."""
    length = s_b - s_a                                 # length of the piece
    s = s_a + length*params["nodes"]                   # nodes mapped from [0, 1] to [s_a, s_b]
    log_q = params["log_weights"] + np.log(length) + log_likelihood(e - s, density, params)
    return s, log_q


def piece_in_z(e, s_a, s_b, params):
    """Gauss-Legendre nodes on [s_a, s_b] in the variable z = (|e_t - s|/alpha)^beta, where the
    cusp of the likelihood at s = e_t becomes the smooth e^{-z}. Same output as piece_in_s."""
    alpha = params["alpha"]
    beta = params["beta"]
    side = 1.0 if s_a >= e else -1.0                   # the piece lies above or below e_t
    z_a = (abs(e - s_a)/alpha)**beta                   # z at the two ends of the piece
    z_b = (abs(e - s_b)/alpha)**beta
    z_low = min(z_a, z_b)
    length = max(z_a, z_b) - z_low                     # length of the piece in z
    z = z_low + length*params["nodes"]                 # nodes mapped from [0, 1] to the piece
    log_z = np.log(z)
    s = e + side*alpha*np.exp(log_z/beta)              # s = e_t + side alpha z^(1/beta)
    # log of weight x length x ds/dz x likelihood, with ds/dz = (alpha/beta) z^(1/beta - 1)
    # and the likelihood e^{-z}
    log_q = params["log_weights"] + np.log(length*alpha/beta) + (1/beta - 1)*log_z - z
    return s, log_q


def chi_quadrature(e, sigma, density, params):
    """chi_t(e_t) and chi'_t(e_t) of eq. (30), from the mean and variance of the scalar
    posterior (21), by Gauss-Legendre quadrature. Same output as chi_laplacian."""
    # Cuts at the peak of the prior (0), the peak of the likelihood (e_t) and the ends of the
    # prior (+-L sigma), so that both peaks sit at the ends of pieces, where the nodes crowd.
    cuts = sorted([-L_WINDOW*sigma, 0.0, e, L_WINDOW*sigma])
    s_all = []                                         # nodes in s of every piece
    log_q_all = []                                     # log of their weights, eq. (21) times ds
    for piece in range(3):
        s_a = cuts[piece]
        s_b = cuts[piece + 1]
        if s_b <= s_a:                                 # e_t on a cut: this piece is empty
            continue
        if density == "gg" and params["beta"] < 1:
            s, log_q = piece_in_z(e, s_a, s_b, params)
        else:
            s, log_q = piece_in_s(e, s_a, s_b, density, params)
        s_all.append(s)
        log_q_all.append(log_q - s*s/(2*sigma**2))     # times the prior N(s; 0, sigma^2), eq. (21)
    s = np.concatenate(s_all)
    log_q = np.concatenate(log_q_all)
    q = np.exp(log_q - log_q.max())                    # largest weight becomes 1, nothing underflows
    total = q.sum()
    mean = (q @ s)/total                               # E[s_t | y_1:t]
    var = (q @ (s - mean)**2)/total                    # Var[s_t | y_1:t]
    return mean, 1 - var/sigma**2                      # chi and chi', eq. (30)


def density_params(density, shape, scale, n_nodes):
    """Everything chi_quadrature needs about one density: shape, scale and the nodes."""
    nodes, weights = np.polynomial.legendre.leggauss(n_nodes)   # on [-1, 1], computed once
    nodes = (nodes + 1)/2                              # moved to [0, 1]: a piece [a, b] takes
    weights = weights/2                                # a + (b - a) node, weight (b - a) weight
    if density == "gg":
        return {"beta": shape, "alpha": scale, "nodes": nodes, "log_weights": np.log(weights)}
    return {"nu": shape, "scale": scale, "nodes": nodes, "log_weights": np.log(weights)}
# === end of the copied block ===


N_NODES = 100               # nodes per piece, as notebook 09
ALPHA_STAR = gg_scale(BETA, VAR_ETA)                   # the true scale of the simulated noise
PARAMS_GG = density_params("gg", BETA, ALPHA_STAR, N_NODES)


def looped(params):
    """chi_quadrature takes one error at a time; this runs it over arrays, with the signature of
    chi_laplacian so the fKF can take any of the three corrections. b_eta is not used."""
    def chi_fn(e, sigma, b_eta=None):
        e_arr, s_arr = np.broadcast_arrays(np.asarray(e, float), np.asarray(sigma, float))
        out = np.array([chi_quadrature(ei, si, "gg", params)
                        for ei, si in zip(e_arr.ravel(), s_arr.ravel())]).reshape(e_arr.shape + (2,))
        return out[..., 0], out[..., 1]
    return chi_fn


# The matched correction: eq. (30) of the draft, from the scalar posterior (21) by quadrature.
chi_matched = looped(PARAMS_GG)                        # GG at beta* and its true scale

noise_scale = np.sqrt(VAR_ETA/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
print(f"matched filter: beta = {BETA}, alpha = {ALPHA_STAR:.6e}   "
      f"(the scale the noise is drawn with: {noise_scale:.6e})")

## 4. The fKF

Eq. (37) (`fKF.mean`) of the draft:

$$\sigma_t^2 = v\,\|\boldsymbol{x}_t\|^2, \qquad
  \boldsymbol{w}_t = \boldsymbol{w}_{t-1} + \frac{\boldsymbol{x}_t}{\|\boldsymbol{x}_t\|^2}\,\chi_t(e_t)$$

$v$ is the only parameter and $\chi_t'$ is not needed, since the variance is fixed. Two forms, as in
`vkf-kf.ipynb`: a **scalar** form with the shared signature `(n, x, d, w0, parameters)`, and a
**batched** form that runs one filter per grid value and realisation, used for the sweeps.

In [ ]:
# Copied from vkf-kf.ipynb, commit 2724d95. sKF_batch is eqs. (35)-(36) of the draft.
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(VAR_THETA_0))
    eps = np.asarray(epsilon, dtype=float)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = w - h
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < L:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - slope/L)
    return mis, k_hist
# === end of the copied block ===


def fKF_joint(n, x, d, w0, parameters):
    """Eq. (37) of the draft (fKF.mean), the fixed family: w_t = w_{t-1} + x_t chi_t(e_t)/||x_t||^2
    with sigma_t^2 = v ||x_t||^2. The correction is a parameter, chi_laplacian by default."""
    v, b_eta = parameters["v"], parameters["b_eta"]
    chi_fn = parameters.get("chi", chi_laplacian)
    L = len(w0)
    w = w0.copy()
    x_t = np.zeros(L)
    w_hist, e = np.zeros((n, L)), np.zeros(n)
    for t in range(n):
        x_t = shift(x[t], x_t)
        e[t] = d[t] - x_t @ w
        w_hist[t] = w
        if t >= L:                                     # waits for a full window, as every notebook
            power = x_t @ x_t                          # ||x_t||^2
            chi, _ = chi_fn(e[t], np.sqrt(v*power), b_eta)
            w = w + x_t*chi/power
    return {"w_hist": w_hist, "e": e}


def fKF_batch(X, D, h, b_eta, v, chi_fn=chi_laplacian, flip_at=None):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < L:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v*power)
        k_hist[:, t] = sigma/b_eta
        chi, _ = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
    return mis, k_hist


print("filters ready")

### Checks

1. The joint fKF, eq. (37) with (43), must be notebook 04's marginal fKF-L (exact), eq. (76)
   (`exact.fKF`) of `main_minorization.tex`: $\Gamma_t \equiv 0$ makes the two the same filter, as
   notebook 07 showed for the sKF. Compared at notebook 04's own `b_eta` and selected $v$.
2. The minorized fKF written here, eq. (37) with (45), must be notebook 04's, eq. (52) (`robust.fKF`)
   of `main_minorization.tex`.
3. The quadrature at $\beta = 1$ is the Laplacian, so the fKF driven by `chi_quadrature` at $\beta = 1$ must
   match the closed form. This checks the machinery of the matched filter on a case with a known answer.
4. Batched and scalar forms must agree to roundoff, for all three corrections.

In [ ]:
# "The draft" in the two docstrings below is the old main_minorization.tex: robust.fKF is its
# eq. (52), exact.fKF its eq. (76).
# === IGNACIO: notebook 04's fKF-L, marginal form - copied verbatim, NOT edited ===
# source: branch ignacio/skf-l-exact, notebooks/04_fkf_base.ipynb, commit db4b3a9
SIGN = np.array([1.0, -1.0])                           # the two branches, varsigma = +1 and -1


def softmax_two(log_weights):
    """Softmax of the two log weights: subtract the largest, exponentiate, normalise."""
    weights = np.exp(log_weights - log_weights.max())  # largest becomes 1, nothing overflows
    return weights/weights.sum()                       # sums to 1


def fKF_L_minorized_closed_form(N, x, d, w0, parameters):
    """Eq. (robust.fKF) of the draft: the Gaussian closed form above with v_eta -> b_eta |e_t|."""
    v = parameters["v"]                                # v, fixed variance on each weight
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    L = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    x_t = np.zeros(L)                                  # x_t, window with the last L samples
    w_hist = np.zeros((N, L))                          # weights at every step
    e = np.zeros((N,))                                 # e_t, prediction error at every step
    gain_hist = np.zeros((N,))                         # gain on x_t e_t at every step

    for k in range(N):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= L:                                     # Augusto waits for a full window; same here
            power = x_t @ x_t                          # ||x_t||^2
            gain = v/(b_eta*abs(e[k]) + v*power)       # g = v / (b_eta |e_t| + v ||x||^2)
            w = w + gain*x_t*e[k]                      # w = w + g x_t e_t
            gain_hist[k] = gain

    return {"h": w, "e": e, "w_hist": w_hist, "gain_hist": gain_hist}


def fKF_L_closed_form(N, x, d, w0, parameters):
    """Eq. (exact.fKF) of the draft (fKF-L, exact): the sKF-L of notebook 03 with its variance fixed at v."""
    v = parameters["v"]                                # v, fixed variance on each weight
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((N, M))                          # weights at every step
    e = np.zeros((N,))                                 # e_t, prediction error at every step
    gain_hist = np.zeros((N,))                         # gain on x_t e_t at every step

    for k in range(N):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does

        if k >= M:                                     # Augusto waits for a full window; same here
            norm_x = np.sqrt(x_t @ x_t)                # ||x_t||
            power = norm_x**2                          # ||x_t||^2

            # The two mixture arguments. Global: no dependence on the weight index m.
            kappa = (SIGN*e[k] - v*power/b_eta)/(np.sqrt(v)*norm_x)

            # Mixture weights, in the log domain. Phi(kappa) underflows and e^{2 e / b_eta}
            # overflows, so neither factor may be formed on its own.
            log_Phi = log_ndtr(kappa)                  # log Phi(kappa), accurate in the left tail
            pi = softmax_two(-SIGN*e[k]/b_eta + log_Phi)

            # Inverse Mills ratio h = phi/Phi, also in the log domain.
            log_phi = -0.5*kappa**2 - 0.5*np.log(2*np.pi)
            h = np.exp(log_phi - log_Phi)

            Lambda = np.sum(SIGN*pi)                   # saturating prediction error, in [-1, 1]
            Gamma = np.sum(SIGN*pi*h)

            gain = v*Lambda/b_eta - np.sqrt(v)*Gamma/norm_x
            w = w + gain*x_t                           # w = w + (v L/b_eta - sqrt(v) G/||x||) x
            gain_hist[k] = gain/e[k]                   # the coefficient of x_t, over e_t

    return {"h": w, "e": e, "w_hist": w_hist, "gain_hist": gain_hist}
# === end of the copied block ===


N_CHECK = 20000
V_04 = {"minorized": 1.23e-3, "exact": 9.38e-4}        # the v notebook 04 selected at -20 dB
x_c, d_c = generate_signals(AR_A, 0, N_CHECK)
w0 = np.zeros(M)

print(f"1. joint fKF against notebook 04's marginal fKF-L (exact), b_eta = {B_VAR:.5f}, "
      f"v = {V_04['exact']:.2e}, {N_CHECK} steps")
t0 = time.time()
w_joint = fKF_joint(N_CHECK, x_c, d_c, w0, {"v": V_04["exact"], "b_eta": B_VAR})["w_hist"]
us_joint = (time.time() - t0)/N_CHECK*1e6
t0 = time.time()
w_marg = fKF_L_closed_form(N_CHECK, x_c, d_c, w0, {"v": V_04["exact"], "b_eta": B_VAR})["w_hist"]
us_marg = (time.time() - t0)/N_CHECK*1e6
print(f"   max |w_joint - w_marginal| = {np.abs(w_joint - w_marg).max():.3e}   "
      f"(max |w| = {np.abs(w_marg).max():.3f})")

print(f"\n2. minorized fKF against notebook 04's, v = {V_04['minorized']:.2e}")
t0 = time.time()
w_min = fKF_joint(N_CHECK, x_c, d_c, w0,
                  {"v": V_04["minorized"], "b_eta": B_VAR, "chi": chi_minorized})["w_hist"]
us_min = (time.time() - t0)/N_CHECK*1e6
w_min04 = fKF_L_minorized_closed_form(N_CHECK, x_c, d_c, w0,
                                      {"v": V_04["minorized"], "b_eta": B_VAR})["w_hist"]
print(f"   max |w_here - w_notebook04| = {np.abs(w_min - w_min04).max():.3e}")

N_Q = 4000
print(f"\n3. the quadrature at beta = 1 against the closed form, {N_Q} steps")
chi_quad_laplace = looped(density_params("gg", 1.0, B_VAR, N_NODES))   # GG, beta = 1, scale b
t0 = time.time()
w_q = fKF_joint(N_Q, x_c, d_c, w0, {"v": V_04["exact"], "b_eta": B_VAR, "chi": chi_quad_laplace})["w_hist"]
us_quad = (time.time() - t0)/N_Q*1e6
print(f"   max |w_quadrature - w_closed_form| = {np.abs(w_q - w_joint[:N_Q]).max():.3e}")

print("\n4. batched against scalar, 3 realisations of 2000 steps, b_eta = E|eta|, v = 5e-4")
sig = [generate_signals(AR_A, s, 2000) for s in range(3)]
Xt, Dt = np.array([s[0] for s in sig]), np.array([s[1] for s in sig])
for label, chi_fn in (("minorized", chi_minorized), ("exact", chi_laplacian), ("matched", chi_matched)):
    mis_b, _ = fKF_batch(Xt, Dt, ho, B_MAD, np.full(3, 5e-4), chi_fn)
    worst = 0.0
    for r in range(3):
        w_s = fKF_joint(2000, Xt[r], Dt[r], w0, {"v": 5e-4, "b_eta": B_MAD, "chi": chi_fn})["w_hist"]
        mis_s = ((w_s - ho)**2).sum(axis=1)
        worst = max(worst, np.abs(mis_b[r] - mis_s).max()/mis_s.max())
    print(f"   {label:>9}: max relative difference = {worst:.3e}")

print(f"\ncost per step, scalar form: minorized {us_min:.1f} us, exact (joint) {us_joint:.1f} us, "
      f"marginal of notebook 04 {us_marg:.1f} us, quadrature {us_quad:.1f} us")

## 5. At rest

The protocol of `vkf-kf.ipynb` and notebook 03: target first, the parameter by grid search to land
on it (search at $R = 3$, $N = 24000$), then one checked run at $R = 20$, $N = 96000$; floor = mean of
the last quarter, converged = first step within 3 dB of the floor. The fKF sweeps $v$, the sKF
sweeps $\varepsilon$.

**Control:** the sKF, eqs. (35)-(36) with the correction (43), is run through the same code. It
must land on `vkf-kf.ipynb`'s 5787 steps (old `b_eta`) and 4729 steps (`E|eta|`), which shows the
sweep was copied correctly.

The matched filter does not use `b_eta`, so it runs once. Its $k_t$ is still reported against
$b_\eta = \mathrm{E}|\eta|$, so the three can be compared.

In [ ]:
EPS_GRID = np.logspace(-8, -3, 11)          # the sKF control, as vkf-kf.ipynb and notebook 03
V_GRID = np.logspace(-6, -1, 11)            # the fKF: two points per decade over five decades


# Copied from vkf-kf.ipynb, commit 2724d95: steady_state and pick unchanged; sweep takes the
# batched filter, its grid and its correction as arguments, so one sweep serves the sKF control
# and the three fKFs.
def steady_state(misalignment):
    floor = 10*np.log10(misalignment[3*len(misalignment)//4:].mean())
    return floor, int(np.argmax(10*np.log10(misalignment) < floor + 3))


def pick(grid, floors):
    best = int(np.argmin(floors))
    g, f = grid[best:], floors[best:]
    if TARGET_DB > f.max():
        return g[-1], "grid too narrow"
    if TARGET_DB < f.min():
        return g[0], "target not reached"
    order = np.argsort(f)
    return 10**np.interp(TARGET_DB, f[order], np.log10(g)[order]), \
        ("ok" if best > 0 else "ok (optimum at grid edge)")


def sweep(fn, grid, X_search, D_search, X_check, D_check, b_eta, **kw):
    """Grid search then the checked run, for one filter."""
    B = len(grid)
    Xs = np.repeat(X_search, B, axis=0)
    Ds = np.repeat(D_search, B, axis=0)
    mis, _ = fn(Xs, Ds, ho, b_eta, np.tile(grid, len(X_search)), **kw)
    floors = np.array([steady_state(c)[0]
                       for c in mis.reshape(len(X_search), B, -1).mean(axis=0)])
    value, status = pick(grid, floors)
    mis, k_hist = fn(X_check, D_check, ho, b_eta, np.full(len(X_check), value), **kw)
    floor, steps = steady_state(mis.mean(axis=0))
    k = k_hist[:, M:].ravel()
    return dict(value=value, floor=floor, steps=steps, status=status,
                k_med=np.median(k), k_lo=np.percentile(k, 10), k_hi=np.percentile(k, 90),
                curve=mis.mean(axis=0))


def table(results, title):
    print(f"\n{title}")
    print(f"{'filter':>20}{'eps or v':>11}{'floor [dB]':>12}{'steps':>8}"
          f"{'k_t median':>12}{'k_t 10-90%':>18}  status")
    for name, r in results.items():
        span = f"{r['k_lo']:.3f} - {r['k_hi']:.3f}"
        print(f"{name:>20}{r['value']:>11.2e}{r['floor']:>12.2f}{r['steps']:>8d}"
              f"{r['k_med']:>12.3f}{span:>18}   {r['status']}")


search = [generate_signals(AR_A, s, N_SEARCH) for s in range(R_SEARCH)]
check = [generate_signals(AR_A, s, N) for s in range(R)]
Xs, Ds = np.array([s[0] for s in search]), np.array([s[1] for s in search])
Xc, Dc = np.array([s[0] for s in check]), np.array([s[1] for s in check])
print("sweep ready")

In [ ]:
LAPLACIAN_FKF = {"fKF, minorized": chi_minorized, "fKF, exact (joint)": chi_laplacian}
rest = {}
for convention, b in (("variance", B_VAR), ("mad", B_MAD)):
    rest[convention] = {}
    t0 = time.time()
    rest[convention]["sKF (control)"] = sweep(sKF_batch, EPS_GRID, Xs, Ds, Xc, Dc, b)
    for name, chi_fn in LAPLACIAN_FKF.items():
        rest[convention][name] = sweep(fKF_batch, V_GRID, Xs, Ds, Xc, Dc, b, chi_fn=chi_fn)
    print(f"b_eta = {b:.5f} ({convention}): done in {time.time() - t0:.0f} s")

t0 = time.time()
rest["mad"]["fKF, matched"] = sweep(fKF_batch, V_GRID, Xs, Ds, Xc, Dc, B_MAD, chi_fn=chi_matched)
print(f"matched: done in {(time.time() - t0)/60:.1f} min")

table(rest["variance"], f"AR({AR_A}), SNR {SNR_DB:.0f} dB, target {TARGET_DB:.0f} dB, "
                        f"b_eta = sqrt(var_eta/2) = {B_VAR:.5f}")
table(rest["mad"], f"AR({AR_A}), SNR {SNR_DB:.0f} dB, target {TARGET_DB:.0f} dB, "
                   f"b_eta = E|eta| = {B_MAD:.5f}")

In [ ]:
print("control: the sKF through this sweep against vkf-kf.ipynb")
for convention, expected in (("variance", 5787), ("mad", 4729)):
    got = rest[convention]["sKF (control)"]["steps"]
    print(f"   {convention:>8}: {got} steps, vkf-kf.ipynb {expected}   "
          f"{'same' if got == expected else 'DIFFERENT'}")

print("\nsteps to converge at equal floor, exact over minorized")
for convention in ("variance", "mad"):
    r = rest[convention]
    ratio = r["fKF, exact (joint)"]["steps"]/r["fKF, minorized"]["steps"]
    print(f"   {convention:>8}: {r['fKF, exact (joint)']['steps']}/{r['fKF, minorized']['steps']} "
          f"= {ratio:.2f}x")
print("   notebook 04 (old b_eta, its own sweep protocol): 6821/5164 = 1.32x")

r = rest["mad"]
print("\nthe matched fKF at rest, b_eta = E|eta|")
for label in ("fKF, exact (joint)", "fKF, minorized"):
    print(f"   matched / {label[5:]:<14}: {r['fKF, matched']['steps']}/{r[label]['steps']} "
          f"= {r['fKF, matched']['steps']/r[label]['steps']:.2f}x")
print(f"\nthe fKF against the sKF (E|eta|): exact {r['fKF, exact (joint)']['steps']} against "
      f"{r['sKF (control)']['steps']} steps")

## 6. Across a change

Notebook 09's test, copied: runs of $2N$ steps, the response flips sign at the middle,
$\boldsymbol{h}_o \to -\boldsymbol{h}_o$ at $t = N$. Each filter keeps the $v$ it was given at rest
($b_\eta = \mathrm{E}|\eta|$), so all three sit on the same floor before the change.
**Recovered:** first step after the change within 3 dB of the floor at rest.

In [ ]:
N_CHANGE = 2*N              # N steps to converge, the change, N steps to recover
scale_gg = np.sqrt(VAR_ETA/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))   # used by the copy below


# === IGNACIO: generate_signals_change and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 9afa027
def generate_signals_change(seed):
    """One realisation of 2N steps: the response flips sign at step N, theta -> -theta."""
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - AR_A**2)*rng.standard_normal(N_CHANGE + WARMUP)   # driving noise, unit-variance x
    x = lfilter([1.0], [1.0, -AR_A], u)[WARMUP:]               # x_t = AR_A x_{t-1} + u_t
    y = np.convolve(ho, x)[:N_CHANGE]                          # clean output through ho
    y[N:] = -y[N:]                                             # from step N on, through -ho
    eta = gennorm.rvs(BETA, scale=scale_gg, size=N_CHANGE, random_state=rng)
    return x, y + eta, eta


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===


change_signals = [generate_signals_change(seed)[:2] for seed in range(R)]
Xch = np.array([s[0] for s in change_signals])
Dch = np.array([s[1] for s in change_signals])

FKF = {"fKF, minorized": chi_minorized, "fKF, exact (joint)": chi_laplacian,
       "fKF, matched": chi_matched}
change = {}
for name, chi_fn in FKF.items():
    t0 = time.time()
    v = rest["mad"][name]["value"]
    mis, _ = fKF_batch(Xch, Dch, ho, B_MAD, np.full(R, v), chi_fn, flip_at=N)
    change[name] = mis.mean(axis=0)
    print(f"{name:<20} v = {v:.3e}   done in {time.time() - t0:.0f} s")

In [ ]:
recovered = {}
print(f"{'filter':<20}{'floor [dB]':>11}{'before [dB]':>13}  {'converged':<10}{'recovery':>10}{'time [s]':>10}")
for name in FKF:
    floor = rest["mad"][name]["floor"]
    before = 10*np.log10(change[name][3*N//4:N].mean())   # last quarter before the change
    recovered[name] = recovery(change[name], floor)
    steps_text = "-" if recovered[name] < 0 else str(recovered[name])
    time_text = "never" if recovered[name] < 0 else f"{recovered[name]/FS:.2f}"
    print(f"{name:<20}{floor:>11.2f}{before:>13.2f}  {'yes' if before <= floor + 3 else 'NO':<10}"
          f"{steps_text:>10}{time_text:>10}")
print("   floor: section 5, at rest; before: mean of the last quarter before the change, same run")

print()
for label in ("fKF, exact (joint)", "fKF, minorized"):
    m, o = recovered["fKF, matched"], recovered[label]
    if m <= 0 or o <= 0:
        print(f"matched / {label[5:]:<14} recovery: one of the two never recovered, no ratio")
        continue
    print(f"matched / {label[5:]:<14} recovery: {m}/{o} = {m/o:.2f}x")

print("\nfor comparison, matched over Laplacian recovery:")
print("   Leszek, sims/sep18/ggbneg_summary_ar0.0.json (white input, random h, change after 6000):")
print("      fKF 2.14 (exact), 2.22 (minorized);  sKF 1.27 (exact), 1.22 (minorized)")
print("   Ignacio, notebook 09 (this setup): sKF 0.79 (joint), 0.81 (minorized)")
print("   Section 4 of the draft: 1.2 to 2.2")

The 3 dB criterion reads recovery at one level only. The same runs, read at several levels on the
way down: the first step after the change below each level.

In [ ]:
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0]
names = list(FKF)
print(f"{'level [dB]':>18}" + "".join(f"{n[5:]:>18}" for n in names)
      + f"{'matched/exact':>15}{'matched/min':>13}")
for level in LEVELS + ["floor + 3"]:
    steps = {n: (time_to(change[n], level) if level != "floor + 3" else recovered[n]) for n in names}
    text = f"{level:>18}" if isinstance(level, str) else f"{level:>18.0f}"
    print(text + "".join(f"{steps[n]:>18d}" for n in names)
          + f"{steps['fKF, matched']/steps['fKF, exact (joint)']:>15.2f}"
          + f"{steps['fKF, matched']/steps['fKF, minorized']:>13.2f}")
print("   steps after the change; b_eta = E|eta|")

**Figure 1.** Misalignment across the change, time counted from it, $b_\eta = \mathrm{E}|\eta|$. Dotted,
in each curve's colour: the instant it gets back within 3 dB of its floor at rest. Grey, dotted: that
recovery level (the three floors are within 0.1 dB of each other). Black, dashed: the change.

In [ ]:
# Fixed colour per filter, from a palette checked for colour-blind readers (the default orange and
# green of matplotlib are almost the same colour for red-blind readers).
COLOUR = {"fKF, minorized": "#2a78d6", "fKF, exact (joint)": "#eb6834", "fKF, matched": "#1baf7a"}

slowest = max([n for n in recovered.values() if n > 0] + [FS])
t_change = (np.arange(N_CHANGE) - N)/FS
keep = (t_change >= -0.5) & (t_change <= min(N/FS, 2.5*slowest/FS))

fig, ax = plt.subplots(figsize=(9, 4.4), constrained_layout=True)
for name in FKF:
    label = name if recovered[name] < 0 else f"{name}, back in {recovered[name]/FS:.2f} s"
    ax.plot(t_change[keep], 10*np.log10(change[name][keep]), color=COLOUR[name], lw=1.5, label=label)
    if recovered[name] > 0:
        ax.axvline(recovered[name]/FS, color=COLOUR[name], ls=":", lw=1.2)
ax.axvline(0, color="k", ls="--", lw=0.9)
ax.axhline(TARGET_DB, color="0.45", lw=0.8)
ax.text(t_change[keep][-1], TARGET_DB - 0.6, f"target, {TARGET_DB:.0f} dB", ha="right", va="top",
        fontsize=8, color="0.3")
level = np.mean([rest["mad"][n]["floor"] for n in FKF]) + 3
ax.axhline(level, color="0.55", ls=":", lw=0.9)
ax.text(t_change[keep][-1], level + 0.6, "recovery level, floor + 3 dB", ha="right", va="bottom",
        fontsize=8, color="0.3")
ax.set(xlabel="time from the change [s]", ylabel="misalignment [dB]", ylim=(-25, 8),
       title=r"fKF across $h_o \to -h_o$, AR(-0.9), $\beta^* = 0.2$, 5 dB")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="upper right")
plt.show()

The same test at $b_\eta = \sqrt{v_\eta/2}$. The two Laplacian fKFs run again, each at the $v$ it
was given at rest under that convention (section 5). The matched fKF does not use $b_\eta$, so its run
above is reused as it is.

In [ ]:
change_var = {"fKF, matched": change["fKF, matched"]}   # the matched fKF does not use b_eta
floor_var = {"fKF, matched": rest["mad"]["fKF, matched"]["floor"]}
for name, chi_fn in LAPLACIAN_FKF.items():
    t0 = time.time()
    v = rest["variance"][name]["value"]
    mis, _ = fKF_batch(Xch, Dch, ho, B_VAR, np.full(R, v), chi_fn, flip_at=N)
    change_var[name] = mis.mean(axis=0)
    floor_var[name] = rest["variance"][name]["floor"]
    print(f"{name:<20} v = {v:.3e}   done in {time.time() - t0:.0f} s")

recovered_var = {name: recovery(change_var[name], floor_var[name]) for name in FKF}
print(f"\n{'filter':<20}{'floor [dB]':>11}{'recovery':>10}{'time [s]':>10}   b_eta = sqrt(var_eta/2)")
for name in FKF:
    steps_text = "-" if recovered_var[name] < 0 else str(recovered_var[name])
    time_text = "never" if recovered_var[name] < 0 else f"{recovered_var[name]/FS:.2f}"
    print(f"{name:<20}{floor_var[name]:>11.2f}{steps_text:>10}{time_text:>10}")

**Figure 2.** As Figure 1, at $b_\eta = \sqrt{v_\eta/2}$. The matched curve is the same run as in
Figure 1. Dotted, in each curve's colour: the instant it gets back within 3 dB of its floor at rest.
Grey, dotted: that recovery level. Black, dashed: the change.

In [ ]:
slowest = max([n for n in recovered_var.values() if n > 0] + [FS])
keep = (t_change >= -0.5) & (t_change <= min(N/FS, 2.5*slowest/FS))

fig, ax = plt.subplots(figsize=(9, 4.4), constrained_layout=True)
for name in FKF:
    label = name if recovered_var[name] < 0 else f"{name}, back in {recovered_var[name]/FS:.2f} s"
    ax.plot(t_change[keep], 10*np.log10(change_var[name][keep]), color=COLOUR[name], lw=1.5, label=label)
    if recovered_var[name] > 0:
        ax.axvline(recovered_var[name]/FS, color=COLOUR[name], ls=":", lw=1.2)
ax.axvline(0, color="k", ls="--", lw=0.9)
ax.axhline(TARGET_DB, color="0.45", lw=0.8)
ax.text(t_change[keep][-1], TARGET_DB - 0.6, f"target, {TARGET_DB:.0f} dB", ha="right", va="top",
        fontsize=8, color="0.3")
level = np.mean(list(floor_var.values())) + 3
ax.axhline(level, color="0.55", ls=":", lw=0.9)
ax.text(t_change[keep][-1], level + 0.6, "recovery level, floor + 3 dB", ha="right", va="bottom",
        fontsize=8, color="0.3")
ax.set(xlabel="time from the change [s]", ylabel="misalignment [dB]", ylim=(-25, 8),
       title=r"fKF across $h_o \to -h_o$, AR(-0.9), $\beta^* = 0.2$, 5 dB, $b_\eta = \sqrt{v_\eta/2}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="upper right")
plt.show()

**Figure 3.** Figures 1 and 2 stacked, on one time axis: top, $b_\eta = \mathrm{E}|\eta|$; bottom,
$b_\eta = \sqrt{v_\eta/2}$. The matched curve is the same run in both panels.

In [ ]:
PANELS = ((r"$b_\eta = \mathrm{E}|\eta|$", change, recovered, {n: rest["mad"][n]["floor"] for n in FKF}),
          (r"$b_\eta = \sqrt{v_\eta/2}$", change_var, recovered_var, floor_var))
slowest = max([n for p in PANELS for n in p[2].values() if n > 0] + [FS])
keep = (t_change >= -0.5) & (t_change <= min(N/FS, 2.5*slowest/FS))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
for ax, (title, curves, back, floors) in zip(axes, PANELS):
    for name in FKF:
        label = name if back[name] < 0 else f"{name}, back in {back[name]/FS:.2f} s"
        ax.plot(t_change[keep], 10*np.log10(curves[name][keep]), color=COLOUR[name], lw=1.5, label=label)
        if back[name] > 0:
            ax.axvline(back[name]/FS, color=COLOUR[name], ls=":", lw=1.2)
    ax.axvline(0, color="k", ls="--", lw=0.9)
    ax.axhline(TARGET_DB, color="0.45", lw=0.8)
    ax.text(t_change[keep][-1], TARGET_DB - 0.6, f"target, {TARGET_DB:.0f} dB", ha="right", va="top",
            fontsize=8, color="0.3")
    level = np.mean(list(floors.values())) + 3
    ax.axhline(level, color="0.55", ls=":", lw=0.9)
    ax.text(t_change[keep][-1], level + 0.6, "recovery level, floor + 3 dB", ha="right", va="bottom",
            fontsize=8, color="0.3")
    ax.set(ylabel="misalignment [dB]", ylim=(-25, 8), title=title)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="upper right")
axes[-1].set_xlabel("time from the change [s]")
fig.suptitle(r"fKF across $h_o \to -h_o$, AR(-0.9), $\beta^* = 0.2$, 5 dB")
plt.show()

## 7. Findings

**Checks.** The joint fKF, eq. (37), is notebook 04's marginal fKF-L (exact), eq. (76) of the old draft,
to 7e-16, one more case of $\Gamma_t \equiv 0$. The minorized fKF matches notebook 04's, eq. (52) of the old
draft, to 6e-16, the quadrature at $\beta = 1$ matches the closed form to 7e-16, and the batched forms
match the scalar ones. The sKF, eqs. (35)-(36), run through this sweep lands on `vkf-kf.ipynb`'s 5787
and 4729 steps exactly, so the protocol was copied correctly.

**At rest.**
- Old `b_eta`: exact over minorized 6842/5168 = **1.32x**, the same as notebook 04 (6821/5164), at almost
  the same $v$.
- `b_eta = E|eta|`: the minorized fKF does not move (5168 -> 5162; its $v$ scales with $b_\eta$), the exact
  fKF gets 21 % faster (6842 -> 5422), and the ratio drops to **1.05x**. The same pattern as the sKF.
- The fKF is slower than the sKF here: 5422 against 4729 steps for the exact filter.
- The matched fKF reaches the floor first: **0.76x** the exact, **0.80x** the minorized. The same as the
  matched sKF of notebook 09 (0.76x).

**Across the change.** At the 3 dB recovery level the matched fKF recovers slightly **faster**:
**0.90x** the exact, **0.94x** the minorized. Leszek's runs give 2.14x and 2.22x. In our setup the fKF does
not bring back the draft's 1.2 to 2.2x.

**But the answer depends on the level.** The matched fKF is slower at the start of the recovery and
catches up only near the floor:

| level | 0 dB | -5 dB | -10 dB | -15 dB | floor + 3 dB |
|---|---|---|---|---|---|
| matched / exact | 1.44 | 1.17 | 1.04 | 0.95 | 0.90 |

The curves cross between -10 and -15 dB, about 0.75 s after the change (Figure 1). That is the mechanism
Section 4 describes: the redescending score treats the first large errors of a change as outliers. Here it
costs time only while the error is large, and the better noise model wins it back near the floor.

**What is left.** Even at 0 dB the ratio (1.44x) is below Leszek's 2.14x, which he reads at floor + 3 dB. So
the level alone does not explain his number; his setup must. For the fKF his runs differ in the input
(white), the response (random, decaying), the time at rest before the change (6000 samples, against
96000 here), and the reading (a 100-sample moving average). The two tests suggested on #8, white input and
the change after 6000 samples, would separate these.

In [ ]:
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")